# Agentic Market Research & Trend Analysis Tutorial

A compact notebook that runs an end-to-end agentic research workflow with:
- `openai`
- `openai-agents`
- `requests`

## Design the agentic research workflow

### Stage 1: Quick Web-Grounded Snapshot
Use the Answer endpoint to collect a focused market summary and source URLs.

### Stage 2: Source Expansion
Select the top 3 sources and scrape them for deeper context.

### Stage 3: Signal Extraction
Extract recurring use cases, positioning, and feature patterns.

### Stage 4: Trend Synthesis
Convert individual signals into higher-level trends and confidence signals.

### Stage 5: Brief Generation
Generate a concise technical research brief and save it as markdown.


### Setup

### Environment
Assumes `OPENAI_API_KEY` and `OLOSTEP_API_KEY` are already configured.


In [2]:
from __future__ import annotations

import json
import os
from typing import Any

import requests
from openai import AsyncOpenAI
from agents import Agent, RunConfig, Runner, function_tool, set_default_openai_client

MODEL_NAME = "gpt-5.2"
INITIAL_TASK = (
    "Research current trends in AI agent tools used by SMB marketing teams. "
    "Focus on recurring use cases, positioning, and common feature patterns."
)
OLOSTEP_BASE_URL = os.getenv("OLOSTEP_BASE_URL", "https://api.olostep.com").rstrip("/")
BRIEF_PATH = "agents_sdk_style_market_research_top3_brief.md"
RESULT_PATH = "agents_sdk_style_market_research_top3_result.json"

set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]))

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {os.environ['OLOSTEP_API_KEY']}",
    "Content-Type": "application/json",
})


## Integrate the Olostep APIs for answer and scraping

In [3]:
def parse_json_object(value: Any) -> dict[str, Any]:
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        text = value.strip()
        if text.startswith("```"):
            text = text.strip("`").replace("json", "", 1).strip()
        return json.loads(text)
    return {}


def unique_http_urls(items: list[Any]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for item in items:
        url = str(item).strip()
        if url.startswith(("http://", "https://")) and url not in seen:
            seen.add(url)
            out.append(url)
    return out


def compact_text(value: Any, limit: int = 7000) -> str:
    return str(value or "").strip()[:limit]


def request_olostep(path: str, payload: dict[str, Any]) -> dict[str, Any]:
    response = session.post(f"{OLOSTEP_BASE_URL}{path}", json=payload, timeout=60)
    response.raise_for_status()
    return response.json()


@function_tool
def olostep_answer_tool(task: str) -> dict[str, Any]:
    """Call Olostep Answer API."""
    return request_olostep("/v1/answers", {"task": task})


@function_tool
def olostep_scrape_tool(url: str) -> dict[str, Any]:
    """Call Olostep Scrape API for one URL."""
    return request_olostep("/v1/scrapes", {"url_to_scrape": url, "formats": ["markdown", "text"]})


## Build a GPT-5.2 research agent with OpenAI Agents SDK

### Agent responsibilities
- `research_agent`: answer + top source scraping package
- `extraction_agent`: structured signal extraction
- `trend_agent`: trend analysis from signals
- `brief_agent`: concise technical brief


In [4]:
research_agent = Agent(
    name="research_agent",
    model=MODEL_NAME,
    tools=[olostep_answer_tool, olostep_scrape_tool],
    instructions=(
        "Always keep INITIAL_TASK central.\n"
        "Run this exact flow:\n"
        "1) Call olostep_answer_tool once with INITIAL_TASK.\n"
        "2) Parse result.json_content and result.sources.\n"
        "3) Select top 3 unique URLs (prefer result.sources order).\n"
        "4) Scrape those top 3 URLs with olostep_scrape_tool.\n"
        "Return strict JSON only with keys: initial_task, answer_summary, answer_json_content, answer_sources, top3_sources, scraped_pages."
    ),
)

extraction_agent = Agent(
    name="extraction_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Extract concrete market signals from provided summary + scraped context only.\n"
        "Return strict JSON with: signals (list of objects).\n"
        "Each signal object: topic, use_case, positioning_pattern, feature_pattern, evidence, source_url."
    ),
)

trend_agent = Agent(
    name="trend_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Analyze recurring patterns from extracted signals.\n"
        "Return strict JSON with: trends (list) and summary (string).\n"
        "Each trend object: trend, why_now, supporting_signals, source_urls, confidence_0_to_1."
    ),
)

brief_agent = Agent(
    name="brief_agent",
    model=MODEL_NAME,
    instructions=(
        "Always include INITIAL_TASK context.\n"
        "Write a concise technical research brief in markdown.\n"
        "Use sections: Executive Summary, Top Trends, Recurring Use Cases, Positioning Patterns, Feature Patterns, Sources."
    ),
)


## Running Market Research Agent


In [5]:
research_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Use tools to complete the flow exactly and return strict JSON only.
"""

research_run = await Runner.run(
    research_agent,
    input=research_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
research_payload = parse_json_object(research_run.final_output)

# 1. Extract answer summary
answer_summary = research_payload.get("answer_summary", "No summary available.")

print("\n=== Agent Answer Summary ===\n")
print(answer_summary)


# 2. Extract top 3 sources (simple, clean)
top3_sources = research_payload.get("top3_sources", [])[:3]

print("\n=== Top 3 Sources ===\n")

if not top3_sources:
    print("No sources found.")
else:
    for i, source in enumerate(top3_sources, start=1):
        print(f"{i}. {source}")



=== Agent Answer Summary ===

SMB marketing teams are adopting “agentic AI” tools that go beyond prompt-based generation to autonomously execute and optimize marketing workflows. Recurring use cases cluster around (1) content production + repurposing, (2) always-on lead capture/qualification and nurture, (3) campaign execution/optimization across channels, and (4) insight-to-action analytics (segmentation, anomaly detection, budget shifts). Positioning commonly frames agents as a 24/7 teammate that helps small teams “do more with less,” emphasizing no/low-code setup, quick time-to-value, and integrations with existing SMB stacks (CRM, email, ads, social). Feature patterns repeat across vendors: natural-language chat interfaces, multi-channel orchestration, real-time data connections to CRM/ad platforms, brand voice guardrails, continuous learning loops from performance signals, and packaged “agents” (e.g., email agent, social agent, lead conversion agent) embedded inside SMB-friendly 

## Extract market signals and analyze trends
Extract signals first, then synthesize trends.


In [6]:
extraction_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Extract signals from this research package. Return strict JSON only.

RESEARCH_PACKAGE:
{json.dumps(research_payload, ensure_ascii=False)}
"""

extraction_run = await Runner.run(
    extraction_agent,
    input=extraction_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
extraction_payload = parse_json_object(extraction_run.final_output)
signals = extraction_payload.get("signals", [])

print("Signals extracted:", len(signals))
# Clean + filter valid signals
valid_signals = [
    s for s in signals
    if s.get("source_url") 
    and not str(s.get("source_url")).startswith("RESEARCH_PACKAGE")
]

top3_signals = valid_signals[:3]

print("\n=== Top 3 Agent Signals ===\n")

for i, s in enumerate(top3_signals, start=1):
    print(f"{i}. {s['topic']}")
    print(f"   Use Case: {s['use_case']}")
    print(f"   Positioning: {s['positioning_pattern']}")
    print(f"   Source: {s['source_url']}\n")

Signals extracted: 15

=== Top 3 Agent Signals ===

1. Content production and repurposing remains a primary SMB agent use case cluster
   Use Case: Automated content generation for blogs/emails/social and repurposing across formats
   Positioning: Tools marketed as helping small teams ship more content faster with minimal effort
   Source: https://circlesstudio.com/blog/best-ai-marketing-tools-small-teams-2025/

2. Always-on lead capture, qualification, and nurture via conversational agents
   Use Case: Lead capture/qualification, routing sales-ready leads, and automated nurturing paths; meeting booking
   Positioning: 24/7 conversational layer that increases conversions without adding headcount
   Source: https://www.creatio.com/glossary/ai-agents-in-marketing

3. Autonomous campaign execution and continuous optimization across channels
   Use Case: Execute and optimize campaigns (email/ads/social), including automated A/B testing and tuning
   Positioning: Always-on optimization and 

In [7]:
trend_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Run trend analysis from the extracted signals. Return strict JSON only.

SIGNALS_JSON:
{json.dumps(signals, ensure_ascii=False)}
"""

trend_run = await Runner.run(
    trend_agent,
    input=trend_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
trend_payload = parse_json_object(trend_run.final_output)
trends = trend_payload.get("trends", [])

print("Trends identified:", len(trends))
print("\n=== Trend Analysis Results ===\n")

if not trends:
    print("No trends identified.")
else:
    print(f"Total Trends Identified: {len(trends)}\n")

    for i, t in enumerate(trends[:3], start=1):
        name = t.get("trend", "Unnamed Trend")
        summary = t.get("why_now", "")
        confidence = t.get("confidence_0_to_1", None)
        supporting = t.get("supporting_signals", [])
        urls = t.get("source_urls", [])[:3]  # keep top 3 links

        print(f"{i}. {name}")
        print(f"   Why now: {summary}")
        if confidence is not None:
            print(f"   Confidence: {confidence}")
        print(f"   Supporting Signals: {len(supporting)} signals")
        if urls:
            print("   Top Sources:")
            for j, u in enumerate(urls, start=1):
                print(f"     {j}. {u}")
        print()

Trends identified: 11

=== Trend Analysis Results ===

Total Trends Identified: 11

1. Shift from prompt-based content generation to agentic, autonomous marketing workflow execution
   Why now: SMB teams are capacity-constrained and increasingly expect AI to move from producing outputs (copy/creatives) to executing multi-step work (launch, monitor, optimize) with minimal human coordination. Tools are being positioned as “teammates” that connect insight-to-action across the funnel.
   Confidence: 0.86
   Supporting Signals: 3 signals
   Top Sources:
     1. RESEARCH_PACKAGE.answer_summary
     2. https://www.creatio.com/glossary/ai-agents-in-marketing

2. Content production + repurposing remains the primary entry use case, increasingly packaged as workflows
   Why now: Content volume requirements across channels keep rising, and SMBs start adoption with the most visible ROI: faster content throughput. Agent tools increasingly bundle generation with adjacent steps (SEO, design, video) to

## Generate the technical research brief
Generate the brief, save markdown + json, and preview output.


In [8]:
brief_prompt = f"""
INITIAL_TASK:
{INITIAL_TASK}

Generate the final concise technical research brief in markdown.

ANSWER_SUMMARY_AND_CONTEXT:
{json.dumps({
    "answer_summary": research_payload.get("answer_summary", ""),
    "top3_sources": research_payload.get("top3_sources", []),
    "scraped_pages": research_payload.get("scraped_pages", []),
}, ensure_ascii=False)}

EXTRACTED_SIGNALS:
{json.dumps(signals, ensure_ascii=False)}

TREND_ANALYSIS:
{json.dumps(trends, ensure_ascii=False)}
"""

brief_run = await Runner.run(
    brief_agent,
    input=brief_prompt,
    run_config=RunConfig(model=MODEL_NAME),
)
final_brief = str(brief_run.final_output).strip()

result = {
    "initial_task": INITIAL_TASK,
    "model": MODEL_NAME,
    "research_payload": research_payload,
    "signals": signals,
    "trends": trends,
    "brief_markdown": final_brief,
}

with open(BRIEF_PATH, "w", encoding="utf-8") as f:
    f.write(final_brief + "\n")

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    f.write(json.dumps(result, indent=2, ensure_ascii=False) + "\n")

print(f"Saved brief to: {BRIEF_PATH}")
print(f"Saved result to: {RESULT_PATH}")
print("\n--- Brief Preview ---\n")
print(final_brief[:3000])


Saved brief to: agents_sdk_style_market_research_top3_brief.md
Saved result to: agents_sdk_style_market_research_top3_result.json

--- Brief Preview ---

## Executive Summary (INITIAL_TASK context)
SMB marketing teams are increasingly adopting **agentic AI tools** that move beyond prompt-based generation to **autonomously execute, monitor, and optimize** multi-step marketing workflows. Recurring adoption patterns center on (1) **content production/repurposing pipelines**, (2) **always-on lead capture and qualification**, (3) **cross-channel campaign execution with continuous experimentation**, and (4) **insight-to-action analytics** (segmentation, recommendations, budget shifts). Tools are commonly positioned as a **“24/7 teammate”** for lean teams, with **fast time-to-value**, **no/low-code setup**, and **deep integrations** into the typical SMB stack (CRM, email, ads, social, chat).

## Top Trends
1. **Shift to autonomous workflow execution (“agentic” marketing)**
   - From single ou

### Workflow Recap and What the Result Means

### Recap
This notebook executes a single agentic pipeline:
1. gather market snapshot and sources,
2. scrape the top sources,
3. extract signals,
4. synthesize trends,
5. produce a concise technical brief.

### Result Objects
- `research_payload`: source-grounded context package from the research stage
- `signals`: atomic evidence points extracted from summary + scraped pages
- `trends`: higher-level patterns synthesized from signal clusters
- `brief_markdown`: final narrative report for decision-making and sharing

### Saved Files
- `agents_sdk_style_market_research_top3_brief.md`: final readable brief
- `agents_sdk_style_market_research_top3_result.json`: full structured output from all stages
